# 06 - Saving and Loading a Fitted UQModel

`UQModel.save()` / `UQModel.load()` persist a fitted model as a small JSON
**checkpoint**: the predictor's task metadata and trained `params`, plus the
uncertainty method's configuration.

What is *not* persisted, on purpose:

- The live backend object -- a PennyLane `QNode` bound to a device, or a
  Qiskit `Sampler`/`Estimator` primitive. These often hold open connections
  (real hardware, IBM Runtime sessions) or simply aren't portable/picklable.
- Any user-supplied `feature_map` / `postprocess` / `bitstring_to_class`
  callables -- these are frequently notebook-local closures or lambdas,
  which plain `pickle` cannot serialize at all.

So `load()` takes the checkpoint plus the live object(s) you reconstruct
yourself (the same `qnode`/`circuit`/`sampler` you already have the code
for) and rebuilds the `UQModel` around them. This is the same "state dict,
not a frozen object graph" pattern used by most ML checkpoint formats, and
it sidesteps both the unpicklable-closures problem and the security risk of
unpickling arbitrary code.

## 1. PennyLane: full save/load round trip

In [1]:
import json

import numpy as np
import pennylane as qml

from quantumuq import ShotBootstrap, wrap_qnode

dev = qml.device("default.qubit", wires=1, shots=1000)


@qml.qnode(dev)
def circuit(x, params):
    qml.RY(x[0] * params[0], wires=0)
    return qml.probs(wires=0)


params = np.array([0.9])
predictor = wrap_qnode(circuit, task="classification", n_classes=2, params=params)
model = predictor.with_uq(ShotBootstrap(n_samples=4, shots=1000, seed=0))

X = np.array([[0.2], [0.8]])
dist_before = model.predict_dist(X)
print("Predictions before saving:\n", dist_before.mean)

Predictions before saving:
 [[0.99225 0.00775]
 [0.88075 0.11925]]


In [2]:
model.save("uqmodel_checkpoint.json")
print(json.dumps(json.loads(open("uqmodel_checkpoint.json").read()), indent=2))

{
  "quantumuq_checkpoint_version": 1,
  "predictor": {
    "backend": "pennylane",
    "task": "classification",
    "n_classes": 2,
    "batched": false,
    "params": [
      0.9
    ]
  },
  "method": {
    "type": "ShotBootstrap",
    "config": {
      "n_samples": 4,
      "shots": 1000,
      "shots_jitter": null,
      "seed": 0
    }
  }
}


Note the checkpoint has no trace of the circuit itself -- only `task`,
`n_classes`, the trained `params` array, and the `ShotBootstrap` config.
Loading requires handing the same `qnode` back in:

In [3]:
from quantumuq.core.predictors import UQModel

loaded = UQModel.load("uqmodel_checkpoint.json", qnode=circuit)

print("params match:      ", np.allclose(loaded.base_predictor.params, params))
print("method config match:", loaded.method.n_samples, loaded.method.shots, loaded.method.seed)

dist_after = loaded.predict_dist(X)
print("\nPredictions after loading:\n", dist_after.mean)
print(
    "\n(Values won't match dist_before exactly -- each call resamples the "
    "circuit's shot noise, the same way calling predict_dist twice on the "
    "original model would also give slightly different numbers.)"
)

params match:       True
method config match: 4 1000 0

Predictions after loading:
 [[0.99125 0.00875]
 [0.865   0.135  ]]

(Values won't match dist_before exactly -- each call resamples the circuit's shot noise, the same way calling predict_dist twice on the original model would also give slightly different numbers.)


In [4]:
# Loading without the live qnode fails loudly instead of silently doing the wrong thing.
try:
    UQModel.load("uqmodel_checkpoint.json")
except ValueError as exc:
    print("Raised as expected:", exc)

Raised as expected: UQModel.load requires qnode=... for a pennylane checkpoint


## 2. `NoiseProfile` checkpoints round-trip the same way

In [5]:
from quantumuq.core.methods import NoiseProfile

model_np = predictor.with_uq(NoiseProfile(sweep_shots=[100, 500, 1000], n_repeats=3))
model_np.save("noise_profile_checkpoint.json")

loaded_np = UQModel.load("noise_profile_checkpoint.json", qnode=circuit)
print("sweep_shots:", loaded_np.method.sweep_shots)
print("n_repeats:  ", loaded_np.method.n_repeats)

sweep_shots: [100, 500, 1000]
n_repeats:   3


## 3. `DeepEnsemble` is explicitly *not* supported

Each ensemble member is itself a live predictor (its own qnode/circuit), so
there is nothing generic to reconstruct it from. Rather than silently
dropping members or saving something broken, `save()` refuses outright:

In [6]:
from quantumuq.core.methods import DeepEnsemble

ensemble_model = UQModel(predictor, DeepEnsemble(predictors=[predictor, predictor]))
try:
    ensemble_model.save("ensemble_checkpoint.json")
except TypeError as exc:
    print("Raised as expected:", exc)

Raised as expected: UQModel.save does not support DeepEnsemble: each ensemble member is itself a live predictor and cannot be captured automatically. Save/load each member's checkpoint individually and rebuild the ensemble by hand.


## 4. Qiskit backends: same checkpoint format

`UQModel.save()`/`.load()` support `wrap_qiskit_sampler` and
`wrap_qiskit_estimator` predictors identically -- the checkpoint just gains
a `"backend": "qiskit_sampler"` (or `"qiskit_estimator"`) tag, and `load()`
expects `sampler=...`/`estimator=...` plus `circuit=...` instead of `qnode=...`.
This uses the V2 `StatevectorSampler` primitive (see notebook 07 for why V2,
not V1, is required).

In [7]:
from qiskit.circuit import Parameter, QuantumCircuit
from qiskit.primitives import StatevectorSampler

from quantumuq import wrap_qiskit_sampler

theta = Parameter("theta")
qc = QuantumCircuit(1)
qc.ry(theta, 0)
qc.measure_all()


def q_feature_map(X_arr: np.ndarray):
    X_arr = np.atleast_2d(X_arr)
    return [[float(x[0])] for x in X_arr]


sampler = StatevectorSampler(seed=np.random.default_rng(0))
q_predictor = wrap_qiskit_sampler(
    sampler, circuit=qc, task="classification", n_classes=2, feature_map=q_feature_map
)
q_model = q_predictor.with_uq(ShotBootstrap(n_samples=4, shots=1000, seed=0))
q_model.save("qiskit_checkpoint.json")
print(json.dumps(json.loads(open("qiskit_checkpoint.json").read()), indent=2))

# Loading needs the same sampler + circuit + feature_map handed back in.
q_loaded = UQModel.load(
    "qiskit_checkpoint.json", sampler=sampler, circuit=qc, feature_map=q_feature_map
)
q_dist = q_loaded.predict_dist(np.array([[1.0]]))
print("\nLoaded Qiskit model predictive mean:", q_dist.mean)

{
  "quantumuq_checkpoint_version": 1,
  "predictor": {
    "backend": "qiskit_sampler",
    "task": "classification",
    "n_classes": 2,
    "params": null
  },
  "method": {
    "type": "ShotBootstrap",
    "config": {
      "n_samples": 4,
      "shots": 1000,
      "shots_jitter": null,
      "seed": 0
    }
  }
}

Loaded Qiskit model predictive mean: [[0.77075 0.22925]]


## Summary

- `UQModel.save(path)` writes a small JSON checkpoint: predictor
  task/params + method config. No live backend objects or user closures.
- `UQModel.load(path, **backend_kwargs)` rebuilds the model given the same
  `qnode` (PennyLane) or `sampler`/`estimator` + `circuit` (Qiskit) you
  already have.
- `ShotBootstrap` and `NoiseProfile` are supported; `DeepEnsemble` raises a
  clear `TypeError` rather than silently doing the wrong thing.
- Tests: `tests/test_persistence.py`.